# 02 - Post-inspection Signal Readiness

Read-only feasibility audit for the BNA-priority Inspection x Claim / Endorsement module.

Reference document: `docs/scoring/post_inspection_signal_readiness_audit.md`

Safety rules for this notebook:

- read from PostgreSQL only;
- write only local CSV quality reports;
- do not modify VHS;
- do not modify Claim Attention Score V1;
- do not create mart tables;
- do not use ML, Isolation Forest, SHAP, or accusatory wording;
- treat DWH technical key `0` as missing.

In [13]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import URL, create_engine, text


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'etl').exists() and (candidate / 'docs').exists():
            return candidate
    raise RuntimeError('Could not locate IRIS_AUTO_FRAUD project root.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'quality_reports' / 'scoring' / 'post_inspection_signals' / 'readiness'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S%z')
MAX_LINK_EXPORT_ROWS = 50000
MAX_VALUE_EXPORT_ROWS = 1000

print(f'[INFO] Project root : {PROJECT_ROOT}')
print(f'[INFO] Output dir   : {OUTPUT_DIR}')

[INFO] Project root : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD
[INFO] Output dir   : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness


## Read-only SQL guard

All database access in this notebook goes through `safe_read_sql()`. The guard blocks common mutating SQL verbs. Local CSV exports are allowed because they are audit artifacts, not database writes.

In [14]:
READ_ONLY_DENY_RE = re.compile(
    r"\b(ALTER|ANALYZE|CALL|COMMENT|COMMIT|COPY|CREATE|DELETE|DROP|EXECUTE|GRANT|INSERT|MERGE|REFRESH|REINDEX|REPLACE|REVOKE|ROLLBACK|TRUNCATE|UPDATE|VACUUM)\b",
    re.IGNORECASE,
)


def strip_sql_comments(sql: str) -> str:
    sql = re.sub(r'/\*.*?\*/', ' ', sql, flags=re.DOTALL)
    sql = re.sub(r'--.*?$', ' ', sql, flags=re.MULTILINE)
    return sql


def safe_read_sql(query: str, engine, params: dict | None = None) -> pd.DataFrame:
    cleaned = strip_sql_comments(query).strip()
    if READ_ONLY_DENY_RE.search(cleaned):
        raise ValueError('Blocked non-read-only SQL. This notebook is audit-only.')
    return pd.read_sql(text(query), engine, params=params)


def parse_env_file(env_path: Path) -> dict[str, str]:
    for enc in ('utf-8', 'cp1252', 'latin-1'):
        try:
            content = env_path.read_text(encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        return {}
    result = {}
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {'\"', "'"}:
            value = value[1:-1]
        result[key.strip()] = value
    return result


def build_read_engine():
    try:
        from etl.utils.runtime import build_engine
        return build_engine()
    except Exception as import_error:
        print(f'[WARN] Could not use etl.utils.runtime.build_engine: {import_error}')
        env_vars = parse_env_file(PROJECT_ROOT / '.env')

        def get_env(key: str, default: str = '') -> str:
            return os.environ.get(key) or env_vars.get(key) or default

        url = URL.create(
            drivername='postgresql+psycopg2',
            host=get_env('DB_HOST', 'localhost'),
            port=int(get_env('DB_PORT', '5432')),
            database=get_env('DB_NAME', 'iris_auto_fraud'),
            username=get_env('DB_USER', 'postgres'),
            password=get_env('DB_PASSWORD', ''),
        )
        return create_engine(url, pool_pre_ping=True)


engine = build_read_engine()
connection_test = safe_read_sql('SELECT 1 AS ok', engine)
print('[OK] Read-only database connection ready.')
display(connection_test)

[OK] Read-only database connection ready.


,ok
0,1


In [15]:
TABLE_STATUS_ROWS: list[dict] = []


def quote_ident(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def qname(schema: str, table: str) -> str:
    return f'{quote_ident(schema)}.{quote_ident(table)}'


def table_exists(schema: str, table: str) -> bool:
    df = safe_read_sql(
        """
        SELECT COUNT(*) AS cnt
        FROM information_schema.tables
        WHERE table_schema = :schema AND table_name = :table
        """,
        engine,
        {'schema': schema, 'table': table},
    )
    return int(df['cnt'].iloc[0]) > 0


def get_columns(schema: str, table: str) -> list[str]:
    df = safe_read_sql(
        """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema AND table_name = :table
        ORDER BY ordinal_position
        """,
        engine,
        {'schema': schema, 'table': table},
    )
    return df['column_name'].tolist()


def select_existing(schema: str, table: str, requested: list[str] | None = None) -> list[str]:
    cols = get_columns(schema, table)
    if requested is None:
        return cols
    return [c for c in requested if c in cols]


def load_optional_table(schema: str, table: str, requested: list[str] | None = None, limit: int | None = None) -> pd.DataFrame:
    if not table_exists(schema, table):
        TABLE_STATUS_ROWS.append({'schema': schema, 'table': table, 'status': 'MISSING', 'rows_loaded': 0, 'columns_loaded': 0})
        print(f'[WARN] Missing table: {schema}.{table}')
        return pd.DataFrame()
    cols = select_existing(schema, table, requested)
    if not cols:
        TABLE_STATUS_ROWS.append({'schema': schema, 'table': table, 'status': 'NO_REQUESTED_COLUMNS', 'rows_loaded': 0, 'columns_loaded': 0})
        print(f'[WARN] No requested columns found: {schema}.{table}')
        return pd.DataFrame()
    sql = f"SELECT {', '.join(quote_ident(c) for c in cols)} FROM {qname(schema, table)}"
    if limit is not None:
        sql += f' LIMIT {int(limit)}'
    df = safe_read_sql(sql, engine)
    TABLE_STATUS_ROWS.append({'schema': schema, 'table': table, 'status': 'LOADED', 'rows_loaded': len(df), 'columns_loaded': len(cols)})
    print(f'[OK] Loaded {schema}.{table}: {len(df):,} rows, {len(cols)} columns')
    return df


def export_report(df: pd.DataFrame, filename: str) -> Path:
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'[EXPORT] {path}')
    return path

## Load required tables

The notebook loads required DWH/mart tables when present. Optional VHS/detail/dimension tables are audited for availability and skipped safely if absent.

In [16]:
inspection_vehicle_cols = [
    'fact_inspection_vehicule_sk', 'inspection_key', 'immatriculation_norm', 'vehicule_sk',
    'date_inspection_sk', 'kilometrage', 'nb_anomalies_total', 'nb_anomalies_critiques',
    'indicateur_anomalie_critique', 'indicateur_inspection_complete', 'agent_controle', 'source_system', 'created_at'
]
inspection_checkpoint_cols = [
    'fact_inspection_checkpoint_sk', 'inspection_checkpoint_key', 'inspection_key', 'vehicule_sk',
    'date_inspection_sk', 'immatriculation_norm', 'zone_controle', 'checkpoint_code', 'checkpoint_libelle',
    'valeur_controle', 'commentaire_zone', 'est_anomalie', 'est_anomalie_critique', 'est_controle_renseigne'
]
vhs_score_cols = [
    'inspection_key', 'vehicule_sk', 'date_inspection_sk', 'immatriculation_norm', 'vhs_score',
    'decision_label', 'decision_code', 'score_run_id', 'created_at'
]
vhs_penalty_cols = [
    'inspection_key', 'vehicule_sk', 'date_inspection_sk', 'immatriculation_norm', 'zone_controle',
    'checkpoint_code', 'checkpoint_libelle', 'valeur_controle', 'est_anomalie', 'est_anomalie_critique',
    'penalty_points', 'score_run_id', 'created_at'
]
claim_cols = [
    'fact_sinistre_sk', 'numero_sinistre', 'code_garantie', 'sinistre_garantie_key', 'sinistre_sk',
    'garantie_sk', 'client_sk', 'contrat_sk', 'vehicule_sk', 'conducteur_sk', 'tiers_sk',
    'date_survenance_sk', 'date_declaration_sk', 'date_ouverture_sk', 'date_cloture_sk',
    'montant_evaluation', 'montant_reglement', 'montant_reserve', 'montant_recours', 'montant_charge_sinistre',
    'motif_cloture_garantie', 'etat_garantie_sinistre'
]
contract_cols = [
    'fact_contrat_sk', 'contrat_mouvement_key', 'contrat_key', 'numero_contrat', 'numero_avenant',
    'numero_mise_a_jour', 'contrat_sk', 'client_sk', 'produit_sk', 'date_debut_contrat_sk',
    'date_fin_contrat_sk', 'date_debut_effet_sk', 'date_fin_effet_sk', 'date_derniere_operation_sk',
    'date_resiliation_sk', 'total_prime', 'nombre_contrat_mouvement', 'est_contrat_actif',
    'est_contrat_resilie', 'est_avenant', 'est_mise_a_jour', 'est_auto_scope', 'situation_contrat',
    'type_resiliation', 'libelle_resiliation'
]
stg_sinistres_cols = [
    'numsnt', 'grntsini', 'grntsini_norm', 'codprod', 'codprod_norm', 'immat', 'numcnt', 'numcnt_norm',
    'idclt', 'causesini', 'causesini_norm', 'motifclot', 'etatgrnt'
]
stg_production_cols = [
    'numcnt', 'numcnt_norm', 'numavt', 'numavt_norm', 'nummaj', 'nummaj_norm', 'codprod', 'codprod_norm',
    'libprdt', 'idclt', 'debcnt', 'debeffet', 'dateprec', 'total_prime', 'situat'
]

stg_inspection_df = load_optional_table('staging', 'stg_inspection')
inspection_vehicle_df = load_optional_table('dwh', 'fact_inspection_vehicule', inspection_vehicle_cols)
inspection_checkpoint_df = load_optional_table('dwh', 'fact_inspection_checkpoint', inspection_checkpoint_cols)
vhs_score_df = load_optional_table('mart', 'fact_vhs_score', vhs_score_cols)
vhs_penalty_df = load_optional_table('mart', 'fact_vhs_penalty_detail', vhs_penalty_cols)

claim_df = load_optional_table('dwh', 'fact_sinistre', claim_cols)
contract_df = load_optional_table('dwh', 'fact_contrat', contract_cols)

dim_vehicule_df = load_optional_table('dwh', 'dim_vehicule', ['vehicule_sk', 'immatriculation', 'vin', 'motorisation'])
dim_garantie_df = load_optional_table('dwh', 'dim_garantie', ['garantie_sk', 'garantie_key', 'code_produit', 'code_garantie', 'libelle_garantie', 'garantie_quality_level'])
dim_sinistre_df = load_optional_table('dwh', 'dim_sinistre', ['sinistre_sk', 'numero_sinistre', 'cause_sinistre', 'libelle_cause_sinistre', 'code_etat', 'indicateur_forcage'])
dim_contrat_df = load_optional_table('dwh', 'dim_contrat', ['contrat_sk', 'contrat_key', 'numero_contrat'])
dim_produit_df = load_optional_table('dwh', 'dim_produit', ['produit_sk', 'code_produit', 'libelle_produit', 'code_famille', 'libelle_famille'])

stg_sinistres_df = load_optional_table('staging', 'stg_sinistres', stg_sinistres_cols)
stg_production_df = load_optional_table('staging', 'stg_production', stg_production_cols)

table_status_df = pd.DataFrame(TABLE_STATUS_ROWS)
display(table_status_df)

[OK] Loaded staging.stg_inspection: 286 rows, 72 columns
[OK] Loaded dwh.fact_inspection_vehicule: 286 rows, 13 columns
[OK] Loaded dwh.fact_inspection_checkpoint: 12,298 rows, 14 columns
[OK] Loaded mart.fact_vhs_score: 2,002 rows, 5 columns
[OK] Loaded mart.fact_vhs_penalty_detail: 33,916 rows, 11 columns
[OK] Loaded dwh.fact_sinistre: 381,893 rows, 22 columns
[OK] Loaded dwh.fact_contrat: 585,514 rows, 25 columns
[OK] Loaded dwh.dim_vehicule: 128,038 rows, 4 columns
[OK] Loaded dwh.dim_garantie: 334 rows, 6 columns
[OK] Loaded dwh.dim_sinistre: 231,496 rows, 6 columns
[OK] Loaded dwh.dim_contrat: 447,585 rows, 3 columns
[OK] Loaded dwh.dim_produit: 37 rows, 5 columns
[OK] Loaded staging.stg_sinistres: 381,893 rows, 12 columns
[OK] Loaded staging.stg_production: 585,514 rows, 15 columns


,schema,table,status,rows_loaded,columns_loaded
0,staging,stg_inspection,LOADED,286,72
1,dwh,fact_inspection_vehicule,LOADED,286,13
2,dwh,fact_inspection_checkpoint,LOADED,12298,14
3,mart,fact_vhs_score,LOADED,2002,5
4,mart,fact_vhs_penalty_detail,LOADED,33916,11
5,dwh,fact_sinistre,LOADED,381893,22
6,dwh,fact_contrat,LOADED,585514,25
7,dwh,dim_vehicule,LOADED,128038,4
8,dwh,dim_garantie,LOADED,334,6
9,dwh,dim_sinistre,LOADED,231496,6


In [17]:
INVALID_TEXT = {'', 'NULL', 'NAN', 'NONE', 'UNKNOWN', 'INCONNU', 'INCONNUE', 'N/A', 'NA', '#N/A', 'NON RENSEIGNE', 'NON RENSEIGNEE'}
TRUE_TEXT = {'1', 'TRUE', 'T', 'Y', 'YES', 'O', 'OUI'}
FALSE_TEXT = {'0', 'FALSE', 'F', 'N', 'NO', 'NON'}


def clean_text_value(value) -> str | None:
    if pd.isna(value):
        return None
    s = str(value).strip().upper()
    if s in INVALID_TEXT:
        return None
    return s


def normalize_code(value) -> str | None:
    s = clean_text_value(value)
    if s is None:
        return None
    try:
        n = float(s.replace(',', '.'))
        if n.is_integer():
            return str(int(n))
    except ValueError:
        pass
    return re.sub(r'\s+', '', s)


def normalize_immat(value) -> str | None:
    s = clean_text_value(value)
    if s is None:
        return None
    s = re.sub(r'[^A-Z0-9]', '', s)
    return s or None


def key_valid(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return values.notna() & values.ne(0)


def safe_date_key_to_datetime(series: pd.Series) -> pd.Series:
    raw = pd.to_numeric(series, errors='coerce')
    text_values = raw.where(raw.notna()).astype('Int64').astype(str).replace({'<NA>': None})
    text_values = text_values.where(text_values.str.fullmatch(r'\d{8}', na=False), None)
    text_values = text_values.where(text_values.ne('0'), None)
    return pd.to_datetime(text_values, format='%Y%m%d', errors='coerce')


def to_bool_series(series: pd.Series) -> pd.Series:
    if series.empty:
        return pd.Series(dtype='boolean')
    s = series.astype(str).str.strip().str.upper()
    return s.isin(TRUE_TEXT)


def delay_band(days: pd.Series) -> pd.Series:
    bins = [-1, 7, 30, 90]
    labels = ['0-7', '8-30', '31-90']
    return pd.cut(days, bins=bins, labels=labels)


def metric_rows(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'run_ts_utc' not in df.columns:
        df.insert(0, 'run_ts_utc', RUN_TS)
    return df


def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def compact_export(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    out = df.head(MAX_LINK_EXPORT_ROWS).copy()
    export_report(out, filename)
    if len(df) > len(out):
        print(f'[WARN] Export capped at {len(out):,} of {len(df):,} rows for {filename}')
    return out


print('[OK] Normalization helpers loaded.')

[OK] Normalization helpers loaded.


In [18]:
inspection = inspection_vehicle_df.copy()
if not inspection.empty:
    inspection['inspection_date'] = safe_date_key_to_datetime(inspection.get('date_inspection_sk', pd.Series(index=inspection.index)))
    inspection['vehicule_sk_valid'] = key_valid(inspection.get('vehicule_sk', pd.Series(index=inspection.index)))
    inspection['immatriculation_norm'] = inspection.get('immatriculation_norm', pd.Series(index=inspection.index)).map(normalize_immat)

checkpoints = inspection_checkpoint_df.copy()
if not checkpoints.empty:
    checkpoints['inspection_date'] = safe_date_key_to_datetime(checkpoints.get('date_inspection_sk', pd.Series(index=checkpoints.index)))
    checkpoints['vehicule_sk_valid'] = key_valid(checkpoints.get('vehicule_sk', pd.Series(index=checkpoints.index)))
    checkpoints['immatriculation_norm'] = checkpoints.get('immatriculation_norm', pd.Series(index=checkpoints.index)).map(normalize_immat)
    checkpoints['is_anomaly'] = to_bool_series(checkpoints.get('est_anomalie', pd.Series(index=checkpoints.index)))
    checkpoints['is_critical_anomaly'] = to_bool_series(checkpoints.get('est_anomalie_critique', pd.Series(index=checkpoints.index)))

defects_by_inspection = pd.DataFrame()
if not checkpoints.empty and 'inspection_key' in checkpoints.columns:
    anomalous = checkpoints[checkpoints['is_anomaly'].fillna(False)].copy()
    if not anomalous.empty:
        defects_by_inspection = (
            anomalous.groupby('inspection_key', dropna=False)
            .agg(
                defective_checkpoint_count=('checkpoint_code', 'count'),
                critical_defective_checkpoint_count=('is_critical_anomaly', 'sum'),
                defective_area=('zone_controle', lambda s: ' | '.join(sorted({str(v) for v in s.dropna().unique()}))),
                defective_checkpoint_label=('checkpoint_libelle', lambda s: ' | '.join(sorted({str(v) for v in s.dropna().unique()})[:8])),
            )
            .reset_index()
        )

claims = claim_df.copy()
if not claims.empty:
    claims = claims.rename(columns={'fact_sinistre_sk': 'claim_sk', 'sinistre_garantie_key': 'claim_business_id'})
    claims['claim_date'] = safe_date_key_to_datetime(claims.get('date_survenance_sk', pd.Series(index=claims.index)))
    claims['declaration_date'] = safe_date_key_to_datetime(claims.get('date_declaration_sk', pd.Series(index=claims.index)))
    for key_col in ['vehicule_sk', 'client_sk', 'contrat_sk', 'garantie_sk', 'sinistre_sk']:
        if key_col in claims.columns:
            claims[f'{key_col}_valid'] = key_valid(claims[key_col])
    claims['numero_sinistre_norm'] = claims.get('numero_sinistre', pd.Series(index=claims.index)).map(normalize_code)
    claims['code_garantie_norm'] = claims.get('code_garantie', pd.Series(index=claims.index)).map(normalize_code)

if not claims.empty and not dim_garantie_df.empty and 'garantie_sk' in claims.columns and 'garantie_sk' in dim_garantie_df.columns:
    claims = claims.merge(dim_garantie_df, on='garantie_sk', how='left', suffixes=('', '_dim_garantie'))

if not claims.empty and not dim_sinistre_df.empty and 'sinistre_sk' in claims.columns and 'sinistre_sk' in dim_sinistre_df.columns:
    claims = claims.merge(dim_sinistre_df, on='sinistre_sk', how='left', suffixes=('', '_dim_sinistre'))

if not claims.empty and not stg_sinistres_df.empty:
    raw_claims = stg_sinistres_df.copy()
    numsnt_col = first_existing(raw_claims, ['numsnt'])
    grnt_col = first_existing(raw_claims, ['grntsini_norm', 'grntsini'])
    if numsnt_col and grnt_col:
        raw_claims['_numero_sinistre_norm'] = raw_claims[numsnt_col].map(normalize_code)
        raw_claims['_code_garantie_norm'] = raw_claims[grnt_col].map(normalize_code)
        if 'immat' in raw_claims.columns:
            raw_claims['claim_immat_norm'] = raw_claims['immat'].map(normalize_immat)
        if 'numcnt_norm' in raw_claims.columns or 'numcnt' in raw_claims.columns:
            contract_col = first_existing(raw_claims, ['numcnt_norm', 'numcnt'])
            raw_claims['claim_contract_key_raw'] = raw_claims[contract_col].map(normalize_code)
        raw_keep = ['_numero_sinistre_norm', '_code_garantie_norm', 'claim_immat_norm', 'claim_contract_key_raw']
        raw_keep = [c for c in raw_keep if c in raw_claims.columns]
        raw_claims = raw_claims[raw_keep].drop_duplicates(['_numero_sinistre_norm', '_code_garantie_norm'])
        claims = claims.merge(
            raw_claims,
            left_on=['numero_sinistre_norm', 'code_garantie_norm'],
            right_on=['_numero_sinistre_norm', '_code_garantie_norm'],
            how='left',
        )

contracts = contract_df.copy()
if not contracts.empty:
    for col in ['contrat_sk', 'client_sk', 'produit_sk']:
        if col in contracts.columns:
            contracts[f'{col}_valid'] = key_valid(contracts[col])
    for date_col in ['date_debut_contrat_sk', 'date_debut_effet_sk', 'date_derniere_operation_sk', 'date_fin_effet_sk']:
        if date_col in contracts.columns:
            contracts[date_col.replace('_sk', '')] = safe_date_key_to_datetime(contracts[date_col])
    contracts['avenant_date'] = contracts.get('date_derniere_operation', pd.Series(pd.NaT, index=contracts.index))
    contracts['movement_date_source'] = 'date_derniere_operation_sk'
    missing_movement = contracts['avenant_date'].isna()
    if 'date_debut_effet' in contracts.columns:
        contracts.loc[missing_movement, 'avenant_date'] = contracts.loc[missing_movement, 'date_debut_effet']
        contracts.loc[missing_movement, 'movement_date_source'] = 'date_debut_effet_sk'
    contracts['is_avenant'] = to_bool_series(contracts.get('est_avenant', pd.Series(index=contracts.index)))
    contracts['is_update'] = to_bool_series(contracts.get('est_mise_a_jour', pd.Series(index=contracts.index)))
    contracts['numero_avenant_num'] = pd.to_numeric(contracts.get('numero_avenant', pd.Series(index=contracts.index)), errors='coerce')
    contracts['numero_mise_a_jour_num'] = pd.to_numeric(contracts.get('numero_mise_a_jour', pd.Series(index=contracts.index)), errors='coerce')
    sort_cols = [c for c in ['contrat_sk', 'avenant_date', 'numero_avenant_num', 'numero_mise_a_jour_num'] if c in contracts.columns]
    contracts = contracts.sort_values(sort_cols, na_position='last') if sort_cols else contracts
    if 'contrat_sk' in contracts.columns:
        contracts['previous_produit_sk'] = contracts.groupby('contrat_sk')['produit_sk'].shift(1) if 'produit_sk' in contracts.columns else np.nan
        contracts['previous_total_prime'] = contracts.groupby('contrat_sk')['total_prime'].shift(1) if 'total_prime' in contracts.columns else np.nan
    if 'produit_sk' in contracts.columns:
        contracts['product_changed_vs_previous'] = key_valid(contracts['produit_sk']) & key_valid(contracts['previous_produit_sk']) & contracts['produit_sk'].ne(contracts['previous_produit_sk'])
    if 'total_prime' in contracts.columns:
        prime = pd.to_numeric(contracts['total_prime'], errors='coerce')
        prev_prime = pd.to_numeric(contracts.get('previous_total_prime', pd.Series(index=contracts.index)), errors='coerce')
        contracts['prime_changed_vs_previous'] = prime.notna() & prev_prime.notna() & prime.ne(prev_prime)

if not contracts.empty and not dim_produit_df.empty and 'produit_sk' in contracts.columns and 'produit_sk' in dim_produit_df.columns:
    contracts = contracts.merge(dim_produit_df, on='produit_sk', how='left', suffixes=('', '_dim_produit'))

print('[OK] Prepared audit frames.')
print(f'  inspections : {len(inspection):,}')
print(f'  checkpoints : {len(checkpoints):,}')
print(f'  claims      : {len(claims):,}')
print(f'  contracts   : {len(contracts):,}')

[OK] Prepared audit frames.
  inspections : 286
  checkpoints : 12,298
  claims      : 381,893
  contracts   : 585,514


## Inspection readiness reports

These reports verify inspection coverage, technical key `0` handling, date parsing, checkpoint areas, and optional VHS availability.

In [19]:
inspection_rows = []
inspection_rows.append({'metric': 'staging_stg_inspection_rows', 'value': len(stg_inspection_df), 'note': 'Raw inspection staging rows loaded if table exists'})
inspection_rows.append({'metric': 'fact_inspection_vehicule_rows', 'value': len(inspection), 'note': 'One row per vehicle inspection'})
inspection_rows.append({'metric': 'fact_inspection_checkpoint_rows', 'value': len(checkpoints), 'note': 'One row per inspection x checkpoint'})
inspection_rows.append({'metric': 'vhs_score_rows_loaded', 'value': len(vhs_score_df), 'note': 'Optional context only; not used as evidence'})
inspection_rows.append({'metric': 'vhs_penalty_detail_rows_loaded', 'value': len(vhs_penalty_df), 'note': 'Optional checkpoint context only'})

if not inspection.empty:
    inspection_rows.extend([
        {'metric': 'inspection_missing_vehicule_sk_zero_or_null', 'value': int((~inspection['vehicule_sk_valid']).sum()), 'note': 'Technical key 0 counted as missing'},
        {'metric': 'inspection_valid_vehicule_sk', 'value': int(inspection['vehicule_sk_valid'].sum()), 'note': 'vehicule_sk nonzero'},
        {'metric': 'inspection_missing_immatriculation_norm', 'value': int(inspection['immatriculation_norm'].isna().sum()), 'note': 'Normalized plate missing'},
        {'metric': 'inspection_valid_date', 'value': int(inspection['inspection_date'].notna().sum()), 'note': 'Parsed YYYYMMDD date key'},
        {'metric': 'inspection_missing_or_invalid_date', 'value': int(inspection['inspection_date'].isna().sum()), 'note': 'Date key 0/invalid/outside parse'},
        {'metric': 'inspection_duplicate_inspection_key', 'value': int(inspection['inspection_key'].duplicated().sum()) if 'inspection_key' in inspection.columns else np.nan, 'note': 'Duplicate inspection business keys'},
    ])

inspection_readiness_summary_df = metric_rows(inspection_rows)
export_report(inspection_readiness_summary_df, 'inspection_readiness_summary.csv')
display(inspection_readiness_summary_df)

if not checkpoints.empty:
    checkpoint_area_summary_df = (
        checkpoints.groupby('zone_controle', dropna=False)
        .agg(
            checkpoint_rows=('inspection_checkpoint_key', 'count'),
            inspections=('inspection_key', pd.Series.nunique),
            distinct_checkpoints=('checkpoint_code', pd.Series.nunique),
            anomaly_rows=('is_anomaly', 'sum'),
            critical_anomaly_rows=('is_critical_anomaly', 'sum'),
            missing_vehicle_key_rows=('vehicule_sk_valid', lambda s: int((~s).sum())),
        )
        .reset_index()
        .sort_values(['anomaly_rows', 'checkpoint_rows'], ascending=False)
    )
else:
    checkpoint_area_summary_df = pd.DataFrame(columns=['zone_controle', 'checkpoint_rows', 'inspections', 'distinct_checkpoints', 'anomaly_rows', 'critical_anomaly_rows', 'missing_vehicle_key_rows'])

export_report(checkpoint_area_summary_df, 'inspection_checkpoint_area_summary.csv')
display(checkpoint_area_summary_df.head(20))

[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\inspection_readiness_summary.csv


,run_ts_utc,metric,value,note
0,2026-07-08 10:21:02+0000,staging_stg_inspection_rows,286,Raw inspection staging rows loaded if table ex...
1,2026-07-08 10:21:02+0000,fact_inspection_vehicule_rows,286,One row per vehicle inspection
2,2026-07-08 10:21:02+0000,fact_inspection_checkpoint_rows,12298,One row per inspection x checkpoint
3,2026-07-08 10:21:02+0000,vhs_score_rows_loaded,2002,Optional context only; not used as evidence
4,2026-07-08 10:21:02+0000,vhs_penalty_detail_rows_loaded,33916,Optional checkpoint context only
5,2026-07-08 10:21:02+0000,inspection_missing_vehicule_sk_zero_or_null,2,Technical key 0 counted as missing
6,2026-07-08 10:21:02+0000,inspection_valid_vehicule_sk,284,vehicule_sk nonzero
7,2026-07-08 10:21:02+0000,inspection_missing_immatriculation_norm,2,Normalized plate missing
8,2026-07-08 10:21:02+0000,inspection_valid_date,284,Parsed YYYYMMDD date key
9,2026-07-08 10:21:02+0000,inspection_missing_or_invalid_date,2,Date key 0/invalid/outside parse


[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\inspection_checkpoint_area_summary.csv


,zone_controle,checkpoint_rows,inspections,distinct_checkpoints,anomaly_rows,critical_anomaly_rows,missing_vehicle_key_rows
3,SOUS_VEHICULE,3718,286,13,451,385,26
4,TOUR_DU_VEHICULE,2574,286,9,301,89,18
1,INTERIEUR,2574,286,9,219,0,18
0,ENTRETIEN,1716,286,6,215,0,12
2,SOUS_CAPOT,1716,286,6,209,24,12


## Claim linkage and damage-area candidates

This section measures whether claims can be linked to inspections by vehicle key or immatriculation, and inventories candidate claim fields that may help map claim nature or damage area.

In [20]:
claim_summary_rows = []
claim_summary_rows.append({'metric': 'fact_sinistre_rows', 'value': len(claims), 'note': 'Central claim table rows'})

if not claims.empty:
    claim_summary_rows.extend([
        {'metric': 'claim_valid_vehicule_sk', 'value': int(claims.get('vehicule_sk_valid', pd.Series(False, index=claims.index)).sum()), 'note': 'vehicule_sk nonzero'},
        {'metric': 'claim_missing_vehicule_sk_zero_or_null', 'value': int((~claims.get('vehicule_sk_valid', pd.Series(False, index=claims.index))).sum()), 'note': 'Technical key 0 counted as missing'},
        {'metric': 'claim_valid_client_sk', 'value': int(claims.get('client_sk_valid', pd.Series(False, index=claims.index)).sum()), 'note': 'client_sk nonzero'},
        {'metric': 'claim_valid_contrat_sk', 'value': int(claims.get('contrat_sk_valid', pd.Series(False, index=claims.index)).sum()), 'note': 'contrat_sk nonzero'},
        {'metric': 'claim_valid_claim_date', 'value': int(claims['claim_date'].notna().sum()), 'note': 'date_survenance_sk parsed as YYYYMMDD'},
        {'metric': 'claim_missing_or_invalid_claim_date', 'value': int(claims['claim_date'].isna().sum()), 'note': 'Date key 0/invalid/outside parse'},
        {'metric': 'claim_with_raw_immatriculation_from_staging', 'value': int(claims.get('claim_immat_norm', pd.Series(index=claims.index)).notna().sum()) if 'claim_immat_norm' in claims.columns else 0, 'note': 'Exploratory read-only bridge via staging.stg_sinistres'},
    ])

    inspection_vehicle_keys = set(inspection.loc[inspection.get('vehicule_sk_valid', pd.Series(False, index=inspection.index)), 'vehicule_sk']) if not inspection.empty and 'vehicule_sk' in inspection.columns else set()
    claim_vehicle_keys = set(claims.loc[claims.get('vehicule_sk_valid', pd.Series(False, index=claims.index)), 'vehicule_sk']) if 'vehicule_sk' in claims.columns else set()
    inspection_immat_keys = set(inspection['immatriculation_norm'].dropna()) if not inspection.empty and 'immatriculation_norm' in inspection.columns else set()
    claim_immat_keys = set(claims['claim_immat_norm'].dropna()) if 'claim_immat_norm' in claims.columns else set()

    claim_summary_rows.extend([
        {'metric': 'common_nonzero_vehicule_sk_count', 'value': len(inspection_vehicle_keys & claim_vehicle_keys), 'note': 'Direct DWH vehicle keys shared by inspection and claim'},
        {'metric': 'common_immatriculation_count', 'value': len(inspection_immat_keys & claim_immat_keys), 'note': 'Normalized plates shared by inspection and raw claim staging'},
    ])

claim_vehicle_linkage_summary_df = metric_rows(claim_summary_rows)
export_report(claim_vehicle_linkage_summary_df, 'claim_vehicle_linkage_summary.csv')
display(claim_vehicle_linkage_summary_df)

candidate_fields = [
    'code_garantie', 'libelle_garantie', 'cause_sinistre', 'libelle_cause_sinistre',
    'motif_cloture_garantie', 'etat_garantie_sinistre', 'code_etat'
]
value_rows = []
for field in candidate_fields:
    if field not in claims.columns:
        value_rows.append({'candidate_field': field, 'candidate_value': None, 'row_count': 0, 'note': 'field_not_available'})
        continue
    counts = claims[field].dropna().astype(str).str.strip().replace('', np.nan).dropna().value_counts().head(100)
    if counts.empty:
        value_rows.append({'candidate_field': field, 'candidate_value': None, 'row_count': 0, 'note': 'no_non_null_values'})
    else:
        for value, count in counts.items():
            value_rows.append({'candidate_field': field, 'candidate_value': value, 'row_count': int(count), 'note': 'top_value'})

claim_damage_area_candidate_values_df = pd.DataFrame(value_rows).head(MAX_VALUE_EXPORT_ROWS)
export_report(claim_damage_area_candidate_values_df, 'claim_damage_area_candidate_values.csv')
display(claim_damage_area_candidate_values_df.head(40))

[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\claim_vehicle_linkage_summary.csv


,run_ts_utc,metric,value,note
0,2026-07-08 10:21:02+0000,fact_sinistre_rows,381893,Central claim table rows
1,2026-07-08 10:21:02+0000,claim_valid_vehicule_sk,377970,vehicule_sk nonzero
2,2026-07-08 10:21:02+0000,claim_missing_vehicule_sk_zero_or_null,3923,Technical key 0 counted as missing
3,2026-07-08 10:21:02+0000,claim_valid_client_sk,381876,client_sk nonzero
4,2026-07-08 10:21:02+0000,claim_valid_contrat_sk,381604,contrat_sk nonzero
5,2026-07-08 10:21:02+0000,claim_valid_claim_date,381893,date_survenance_sk parsed as YYYYMMDD
6,2026-07-08 10:21:02+0000,claim_missing_or_invalid_claim_date,0,Date key 0/invalid/outside parse
7,2026-07-08 10:21:02+0000,claim_with_raw_immatriculation_from_staging,377970,Exploratory read-only bridge via staging.stg_s...
8,2026-07-08 10:21:02+0000,common_nonzero_vehicule_sk_count,231,Direct DWH vehicle keys shared by inspection a...
9,2026-07-08 10:21:02+0000,common_immatriculation_count,231,Normalized plates shared by inspection and raw...


[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\claim_damage_area_candidate_values.csv


,candidate_field,candidate_value,row_count,note
0,code_garantie,CAS,130853,top_value
1,code_garantie,RCM,90315,top_value
2,code_garantie,IDA,54958,top_value
3,code_garantie,BG,28294,top_value
4,code_garantie,REM,17539,top_value
5,code_garantie,ASR,15624,top_value
6,code_garantie,RCC,13501,top_value
7,code_garantie,TR,10280,top_value
8,code_garantie,RCDE,8471,top_value
9,code_garantie,DOC,4941,top_value


### Raw claim immatriculation deep dive

This read-only control profiles `staging.stg_sinistres.immat`, because `dwh.fact_sinistre` does not carry the raw immatriculation and `vehicule_sk` coverage is very weak.

It measures raw coverage, normalized coverage using the same A-Z/0-9 rule as the SQL audit, top normalized values, and normalized length distribution.

Important: do not use `date_key + 90` for delay checks. Date keys are integers formatted as `YYYYMMDD`; delays must be calculated after parsing to real dates or through `dim_date`.


In [21]:
raw_immat_profile_rows = []
raw_top_immat_df = pd.DataFrame(columns=['immat_norm', 'rows'])
raw_immat_length_distribution_df = pd.DataFrame(columns=['immat_length', 'rows', 'distinct_values'])

if not stg_sinistres_df.empty and 'immat' in stg_sinistres_df.columns:
    raw_immat_text = stg_sinistres_df['immat'].astype('string')
    raw_immat_trim = raw_immat_text.str.strip()
    raw_immat_upper = raw_immat_trim.str.upper()
    raw_immat_norm = stg_sinistres_df['immat'].map(normalize_immat)
    non_empty_raw_immat = raw_immat_text.notna() & raw_immat_trim.ne('')
    non_empty_norm = raw_immat_norm[non_empty_raw_immat]
    normalized_available = non_empty_norm.notna() & non_empty_norm.ne('')
    invalid_like_values = {
        '0', '00', '000', '0000', 'NAN', 'NULL', 'NONE',
        'INCONNU', 'NON RENSEIGNE', 'NEANT', 'SANS',
        'PIETON', 'MOBYLETTE', 'NON ASSURE',
    }

    raw_immat_profile_rows.extend([
        {'metric': 'stg_sinistres_total_rows', 'value': len(stg_sinistres_df), 'note': 'Raw claim staging rows'},
        {'metric': 'stg_sinistres_missing_raw_immat', 'value': int((~non_empty_raw_immat).sum()), 'note': 'immat is null or blank'},
        {'metric': 'stg_sinistres_non_empty_raw_immat', 'value': int(non_empty_raw_immat.sum()), 'note': 'immat available before normalization'},
        {'metric': 'stg_sinistres_distinct_raw_immat', 'value': int(raw_immat_upper[non_empty_raw_immat].nunique()), 'note': 'Distinct UPPER(TRIM(immat)) values'},
        {'metric': 'stg_sinistres_distinct_normalized_immat', 'value': int(non_empty_norm[normalized_available].nunique()), 'note': 'Distinct normalized plates after removing non A-Z0-9 characters'},
        {'metric': 'stg_sinistres_empty_after_immat_normalization', 'value': int((~normalized_available).sum()), 'note': 'Non-empty raw values that become empty after normalization'},
        {'metric': 'stg_sinistres_invalid_like_raw_immat', 'value': int(raw_immat_upper[non_empty_raw_immat].isin(invalid_like_values).sum()), 'note': 'Obvious invalid-like values among non-empty raw plates'},
    ])

    if not dim_vehicule_df.empty and {'vehicule_sk', 'immatriculation'}.issubset(dim_vehicule_df.columns):
        dim_immat_norm = dim_vehicule_df['immatriculation'].map(normalize_immat)
        dim_immat_set = set(dim_immat_norm.dropna())
        matched_dim = non_empty_norm[normalized_available].isin(dim_immat_set)
        raw_immat_profile_rows.extend([
            {'metric': 'dim_vehicule_total_rows', 'value': len(dim_vehicule_df), 'note': 'Vehicle dimension rows'},
            {'metric': 'dim_vehicule_zero_key_rows', 'value': int((~key_valid(dim_vehicule_df['vehicule_sk'])).sum()), 'note': 'Technical key 0 counted as missing'},
            {'metric': 'stg_sinistres_normalized_immat_matched_dim_vehicule', 'value': int(matched_dim.sum()), 'note': 'Normalized claim plates matched to dim_vehicule'},
            {'metric': 'stg_sinistres_normalized_immat_unmatched_dim_vehicule', 'value': int((~matched_dim).sum()), 'note': 'Normalized claim plates not present in dim_vehicule'},
        ])
    else:
        raw_immat_profile_rows.append({'metric': 'dim_vehicule_available_for_raw_immat_match', 'value': 0, 'note': 'dim_vehicule missing or required columns unavailable'})

    normalized_values = non_empty_norm[normalized_available]
    raw_top_immat_df = (
        normalized_values
        .value_counts()
        .head(100)
        .rename_axis('immat_norm')
        .reset_index(name='rows')
    )
    tmp_lengths = pd.DataFrame({'immat_norm': normalized_values})
    tmp_lengths['immat_length'] = tmp_lengths['immat_norm'].str.len()
    raw_immat_length_distribution_df = (
        tmp_lengths.groupby('immat_length', dropna=False)
        .agg(rows=('immat_norm', 'size'), distinct_values=('immat_norm', pd.Series.nunique))
        .reset_index()
        .sort_values('immat_length')
    )
else:
    raw_immat_profile_rows.append({'metric': 'stg_sinistres_raw_immat_available', 'value': 0, 'note': 'staging.stg_sinistres.immat unavailable'})

claim_raw_immat_profile_df = metric_rows(raw_immat_profile_rows)
export_report(claim_raw_immat_profile_df, 'claim_raw_immat_profile.csv')
export_report(raw_top_immat_df, 'claim_raw_immat_top_values.csv')
export_report(raw_immat_length_distribution_df, 'claim_raw_immat_length_distribution.csv')
display(claim_raw_immat_profile_df)
display(raw_top_immat_df.head(30))
display(raw_immat_length_distribution_df)


[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\claim_raw_immat_profile.csv
[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\claim_raw_immat_top_values.csv
[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\claim_raw_immat_length_distribution.csv


,run_ts_utc,metric,value,note
0,2026-07-08 10:21:02+0000,stg_sinistres_total_rows,381893,Raw claim staging rows
1,2026-07-08 10:21:02+0000,stg_sinistres_missing_raw_immat,3923,immat is null or blank
2,2026-07-08 10:21:02+0000,stg_sinistres_non_empty_raw_immat,377970,immat available before normalization
3,2026-07-08 10:21:02+0000,stg_sinistres_distinct_raw_immat,128052,Distinct UPPER(TRIM(immat)) values
4,2026-07-08 10:21:02+0000,stg_sinistres_distinct_normalized_immat,128032,Distinct normalized plates after removing non ...
5,2026-07-08 10:21:02+0000,stg_sinistres_empty_after_immat_normalization,0,Non-empty raw values that become empty after n...
6,2026-07-08 10:21:02+0000,stg_sinistres_invalid_like_raw_immat,0,Obvious invalid-like values among non-empty ra...
7,2026-07-08 10:21:02+0000,dim_vehicule_total_rows,128038,Vehicle dimension rows
8,2026-07-08 10:21:02+0000,dim_vehicule_zero_key_rows,0,Technical key 0 counted as missing
9,2026-07-08 10:21:02+0000,stg_sinistres_normalized_immat_matched_dim_veh...,376979,Normalized claim plates matched to dim_vehicule


,immat_norm,rows
0,4639TU204,42
1,3312TU222,40
2,3123TU167,39
3,6156TU212,38
4,4294TU185,38
5,3563TU176,36
6,3572TU194,34
7,1671TU163,34
8,4486TU168,34
9,9122TU115,34


,immat_length,rows,distinct_values
0,5,95,33
1,6,880,355
2,7,8980,3649
3,8,92178,35970
4,9,275587,87896
5,10,161,82
6,11,21,8
7,12,2,1
8,13,4,3
9,14,6,4


## Scenario A - Inspection to claim linkage

Candidate links are restricted to claims after inspection and within 90 days. These are feasibility links only; no attention score is produced here.

In [22]:
def finalize_claim_links(df: pd.DataFrame, method: str, quality: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    df = df.copy()
    df['linkage_method'] = method
    df['linkage_quality'] = quality
    df['days_inspection_to_claim'] = (df['claim_date'] - df['inspection_date']).dt.days
    df = df[df['days_inspection_to_claim'].between(0, 90, inclusive='both')].copy()
    if df.empty:
        return df
    df['delay_band'] = delay_band(df['days_inspection_to_claim']).astype(str)
    if not defects_by_inspection.empty and 'inspection_key' in df.columns:
        df = df.merge(defects_by_inspection, on='inspection_key', how='left')
    df['has_documented_inspection_defect'] = df.get('defective_checkpoint_count', pd.Series(index=df.index)).fillna(0).gt(0)
    keep = [
        'linkage_method', 'linkage_quality', 'inspection_key', 'immatriculation_norm', 'vehicule_sk',
        'claim_sk', 'claim_business_id', 'numero_sinistre', 'code_garantie', 'libelle_garantie',
        'client_sk', 'contrat_sk', 'inspection_date', 'claim_date', 'days_inspection_to_claim', 'delay_band',
        'has_documented_inspection_defect', 'defective_checkpoint_count', 'critical_defective_checkpoint_count',
        'defective_area', 'defective_checkpoint_label', 'motif_cloture_garantie', 'etat_garantie_sinistre',
        'cause_sinistre', 'libelle_cause_sinistre'
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].sort_values(['days_inspection_to_claim', 'inspection_key', 'claim_sk'], na_position='last')


claim_link_frames = []

if not inspection.empty and not claims.empty and 'vehicule_sk' in inspection.columns and 'vehicule_sk' in claims.columns:
    insp_v = inspection[inspection['vehicule_sk_valid'] & inspection['inspection_date'].notna()].copy()
    claim_v = claims[claims.get('vehicule_sk_valid', pd.Series(False, index=claims.index)) & claims['claim_date'].notna()].copy()
    if not insp_v.empty and not claim_v.empty:
        link_v = insp_v.merge(claim_v, on='vehicule_sk', how='inner', suffixes=('', '_claim'))
        claim_link_frames.append(finalize_claim_links(link_v, 'VEHICULE_SK_NONZERO', 'HIGH'))

if not inspection.empty and not claims.empty and 'claim_immat_norm' in claims.columns:
    insp_i = inspection[inspection['immatriculation_norm'].notna() & inspection['inspection_date'].notna()].copy()
    claim_i = claims[claims['claim_immat_norm'].notna() & claims['claim_date'].notna()].copy()
    if not insp_i.empty and not claim_i.empty:
        link_i = insp_i.merge(claim_i, left_on='immatriculation_norm', right_on='claim_immat_norm', how='inner', suffixes=('', '_claim'))
        claim_link_frames.append(finalize_claim_links(link_i, 'IMMATRICULATION_FROM_STAGING', 'MEDIUM'))

if not inspection.empty and not claims.empty and {'contrat_sk', 'client_sk'}.issubset(inspection.columns):
    insp_f = inspection[inspection['inspection_date'].notna()].copy()
    claim_f = claims[claims['claim_date'].notna()].copy()
    if 'contrat_sk' in insp_f.columns and 'contrat_sk' in claim_f.columns:
        link_c = insp_f.merge(claim_f, on='contrat_sk', how='inner', suffixes=('', '_claim'))
        claim_link_frames.append(finalize_claim_links(link_c, 'CONTRACT_FALLBACK_IF_AVAILABLE', 'LOW'))

inspection_to_claim_candidate_links_df = pd.concat([f for f in claim_link_frames if not f.empty], ignore_index=True) if claim_link_frames else pd.DataFrame()
if not inspection_to_claim_candidate_links_df.empty:
    inspection_to_claim_candidate_links_df = inspection_to_claim_candidate_links_df.drop_duplicates(['linkage_method', 'inspection_key', 'claim_sk'])

compact_export(inspection_to_claim_candidate_links_df, 'inspection_to_claim_candidate_links.csv')
display(inspection_to_claim_candidate_links_df.head(30))

if not inspection_to_claim_candidate_links_df.empty:
    scenario_a_summary_df = (
        inspection_to_claim_candidate_links_df.groupby(['linkage_method', 'linkage_quality', 'delay_band'], dropna=False)
        .agg(candidate_links=('claim_sk', 'count'), inspections=('inspection_key', pd.Series.nunique), claims=('claim_sk', pd.Series.nunique), links_with_defects=('has_documented_inspection_defect', 'sum'))
        .reset_index()
    )
else:
    scenario_a_summary_df = pd.DataFrame(columns=['linkage_method', 'linkage_quality', 'delay_band', 'candidate_links', 'inspections', 'claims', 'links_with_defects'])

display(scenario_a_summary_df)

[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\inspection_to_claim_candidate_links.csv


,linkage_method,linkage_quality,inspection_key,immatriculation_norm,vehicule_sk,claim_sk,claim_business_id,numero_sinistre,code_garantie,libelle_garantie,...,delay_band,has_documented_inspection_defect,defective_checkpoint_count,critical_defective_checkpoint_count,defective_area,defective_checkpoint_label,motif_cloture_garantie,etat_garantie_sinistre,cause_sinistre,libelle_cause_sinistre
0,VEHICULE_SK_NONZERO,HIGH,8188TU252|20260401|240,8188TU252,98184,379455,S26521000000176|CAS,S26521000000176,CAS,CONTRE ASSURANCE SPECIALE,...,0-7,False,NaN,NaN,NaN,NaN,NaN,NaN,10,DOMMAGE COLLISION
1,VEHICULE_SK_NONZERO,HIGH,8188TU252|20260401|240,8188TU252,98184,379456,S26521000000176|DOC,S26521000000176,DOC,DOMMAGES COLLISION,...,0-7,False,NaN,NaN,NaN,NaN,NaN,NaN,10,DOMMAGE COLLISION
2,VEHICULE_SK_NONZERO,HIGH,8188TU252|20260401|240,8188TU252,98184,379457,S26521000000176|RCM,S26521000000176,RCM,RESPONSABILITE CIVILE MATERIELLE,...,0-7,False,NaN,NaN,NaN,NaN,NaN,NaN,10,DOMMAGE COLLISION
3,VEHICULE_SK_NONZERO,HIGH,8185TU96|20251111|140,8185TU96,98148,358732,S25511000024129|BG,S25511000024129,BG,BRIS DE GLACE,...,0-7,True,21.0,0.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | SOUS_VEHI...,Controle bougies d allumage | Controle du nive...,NaN,NaN,11,BRIS DE GLACE
4,VEHICULE_SK_NONZERO,HIGH,8185TU96|20251111|140,8185TU96,98148,358733,S25511000024129|CAS,S25511000024129,CAS,CONTRE ASSURANCE SPECIALE,...,0-7,True,21.0,0.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | SOUS_VEHI...,Controle bougies d allumage | Controle du nive...,NaN,NaN,11,BRIS DE GLACE
5,VEHICULE_SK_NONZERO,HIGH,2894TU107|20251014|78,2894TU107,25786,365800,S25521000020053|ASR,S25521000020053,ASR,AVANCE SUR RECOURS,...,0-7,True,9.0,0.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | TOUR_DU_V...,Controle du niveau du liquide de refroidi | Co...,NaN,NaN,5,CX RISQUES DIVERS
6,VEHICULE_SK_NONZERO,HIGH,2894TU107|20251014|78,2894TU107,25786,365801,S25521000020053|CAS,S25521000020053,CAS,CONTRE ASSURANCE SPECIALE,...,0-7,True,9.0,0.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | TOUR_DU_V...,Controle du niveau du liquide de refroidi | Co...,NaN,NaN,5,CX RISQUES DIVERS
7,VEHICULE_SK_NONZERO,HIGH,6485TU141|20250731|155,6485TU141,74852,346012,S25511000009695|RCM,S25511000009695,RCM,RESPONSABILITE CIVILE MATERIELLE,...,0-7,True,4.0,2.0,SOUS_VEHICULE | TOUR_DU_VEHICULE,Controle etat approfondi et mise a p 1 | Contr...,NaN,NaN,4,CX TRANSPORT
8,VEHICULE_SK_NONZERO,HIGH,4790TU168|20260326|227,4790TU168,51642,376744,S26511000001137|CAS,S26511000001137,CAS,CONTRE ASSURANCE SPECIALE,...,0-7,True,2.0,1.0,SOUS_VEHICULE,Controle etancheite des amortisseurs a | Contr...,NaN,NaN,3,DECES
9,VEHICULE_SK_NONZERO,HIGH,4790TU168|20260326|227,4790TU168,51642,376745,S26511000001137|IDA,S26511000001137,IDA,INDEMNISATION DIRECTE DES ASSURES,...,0-7,True,2.0,1.0,SOUS_VEHICULE,Controle etancheite des amortisseurs a | Contr...,NaN,NaN,3,DECES


,linkage_method,linkage_quality,delay_band,candidate_links,inspections,claims,links_with_defects
0,IMMATRICULATION_FROM_STAGING,MEDIUM,0-7,10,5,10,7
1,IMMATRICULATION_FROM_STAGING,MEDIUM,31-90,33,16,33,25
2,IMMATRICULATION_FROM_STAGING,MEDIUM,8-30,18,11,17,18
3,VEHICULE_SK_NONZERO,HIGH,0-7,10,5,10,7
4,VEHICULE_SK_NONZERO,HIGH,31-90,33,16,33,25
5,VEHICULE_SK_NONZERO,HIGH,8-30,18,11,17,18


## Scenario B - Inspection to contract movement / endorsement linkage

Candidate links are restricted to contract movements after inspection and within 90 days. Weak exploratory routes are labelled as low confidence.

In [23]:
contract_rows = []
contract_rows.append({'metric': 'fact_contrat_rows', 'value': len(contracts), 'note': 'Contract movement fact rows'})
if not contracts.empty:
    contract_rows.extend([
        {'metric': 'valid_contrat_sk', 'value': int(contracts.get('contrat_sk_valid', pd.Series(False, index=contracts.index)).sum()), 'note': 'contrat_sk nonzero'},
        {'metric': 'missing_contrat_sk_zero_or_null', 'value': int((~contracts.get('contrat_sk_valid', pd.Series(False, index=contracts.index))).sum()), 'note': 'Technical key 0 counted as missing'},
        {'metric': 'avenant_rows_est_avenant', 'value': int(contracts.get('is_avenant', pd.Series(False, index=contracts.index)).sum()), 'note': 'Rows flagged as endorsement'},
        {'metric': 'update_rows_est_mise_a_jour', 'value': int(contracts.get('is_update', pd.Series(False, index=contracts.index)).sum()), 'note': 'Rows flagged as updates'},
        {'metric': 'valid_avenant_or_effect_date', 'value': int(contracts['avenant_date'].notna().sum()), 'note': 'date_derniere_operation_sk, falling back to date_debut_effet_sk'},
        {'metric': 'product_changed_vs_previous_rows', 'value': int(contracts.get('product_changed_vs_previous', pd.Series(False, index=contracts.index)).sum()), 'note': 'Observable product change, not guarantee-level proof'},
        {'metric': 'prime_changed_vs_previous_rows', 'value': int(contracts.get('prime_changed_vs_previous', pd.Series(False, index=contracts.index)).sum()), 'note': 'Observable prime change, not guarantee-level proof'},
        {'metric': 'dim_produit_rows_loaded', 'value': len(dim_produit_df), 'note': 'Product labels available if table exists'},
        {'metric': 'staging_stg_production_rows_loaded', 'value': len(stg_production_df), 'note': 'Raw production rows loaded if table exists'},
    ])

contract_avenant_readiness_summary_df = metric_rows(contract_rows)
export_report(contract_avenant_readiness_summary_df, 'contract_avenant_readiness_summary.csv')
display(contract_avenant_readiness_summary_df)


def finalize_avenant_links(df: pd.DataFrame, method: str, quality: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    df = df.copy()
    df['linkage_method'] = method
    df['linkage_quality'] = quality
    df['days_inspection_to_avenant'] = (df['avenant_date'] - df['inspection_date']).dt.days
    df = df[df['days_inspection_to_avenant'].between(0, 90, inclusive='both')].copy()
    if df.empty:
        return df
    df['delay_band'] = delay_band(df['days_inspection_to_avenant']).astype(str)
    if not defects_by_inspection.empty and 'inspection_key' in df.columns:
        df = df.merge(defects_by_inspection, on='inspection_key', how='left')
    df['has_documented_inspection_defect'] = df.get('defective_checkpoint_count', pd.Series(index=df.index)).fillna(0).gt(0)
    df['coverage_change_observable_flag'] = df.get('product_changed_vs_previous', pd.Series(False, index=df.index)).fillna(False) | df.get('prime_changed_vs_previous', pd.Series(False, index=df.index)).fillna(False)
    keep = [
        'linkage_method', 'linkage_quality', 'inspection_key', 'immatriculation_norm', 'vehicule_sk',
        'fact_contrat_sk', 'contrat_sk', 'contrat_mouvement_key', 'contrat_key', 'numero_contrat',
        'numero_avenant', 'numero_mise_a_jour', 'client_sk', 'produit_sk', 'code_produit', 'libelle_produit',
        'inspection_date', 'avenant_date', 'movement_date_source', 'days_inspection_to_avenant', 'delay_band',
        'is_avenant', 'is_update', 'product_changed_vs_previous', 'prime_changed_vs_previous',
        'coverage_change_observable_flag', 'has_documented_inspection_defect', 'defective_checkpoint_count',
        'critical_defective_checkpoint_count', 'defective_area', 'defective_checkpoint_label'
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].sort_values(['days_inspection_to_avenant', 'inspection_key', 'fact_contrat_sk'], na_position='last')


avenant_link_frames = []

if not inspection.empty and not contracts.empty and 'contrat_sk' in inspection.columns:
    insp_c = inspection[inspection['inspection_date'].notna() & key_valid(inspection['contrat_sk'])].copy()
    contract_c = contracts[contracts.get('contrat_sk_valid', pd.Series(False, index=contracts.index)) & contracts['avenant_date'].notna()].copy()
    if not insp_c.empty and not contract_c.empty:
        direct_links = insp_c.merge(contract_c, on='contrat_sk', how='inner', suffixes=('', '_contract'))
        avenant_link_frames.append(finalize_avenant_links(direct_links, 'INSPECTION_CONTRACT_SK_DIRECT', 'HIGH'))

if not inspection.empty and not claims.empty and not contracts.empty and 'claim_immat_norm' in claims.columns:
    # Weak exploratory route: inspection plate -> claim raw plate -> claim contract -> contract movement.
    # This is for feasibility measurement only and should not create attention points.
    insp_i = inspection[inspection['inspection_date'].notna() & inspection['immatriculation_norm'].notna()].copy()
    claim_contracts = claims[
        claims['claim_immat_norm'].notna()
        & claims.get('contrat_sk_valid', pd.Series(False, index=claims.index))
    ][['claim_immat_norm', 'contrat_sk', 'client_sk']].drop_duplicates()
    contract_movements = contracts[
        contracts.get('contrat_sk_valid', pd.Series(False, index=contracts.index))
        & contracts['avenant_date'].notna()
        & (contracts.get('is_avenant', pd.Series(False, index=contracts.index)) | contracts.get('is_update', pd.Series(False, index=contracts.index)))
    ].copy()
    if not insp_i.empty and not claim_contracts.empty and not contract_movements.empty:
        plate_to_contract = insp_i.merge(claim_contracts, left_on='immatriculation_norm', right_on='claim_immat_norm', how='inner', suffixes=('', '_claim'))
        weak_links = plate_to_contract.merge(contract_movements, on='contrat_sk', how='inner', suffixes=('', '_contract'))
        avenant_link_frames.append(finalize_avenant_links(weak_links, 'IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK', 'LOW'))

inspection_to_avenant_candidate_links_df = pd.concat([f for f in avenant_link_frames if not f.empty], ignore_index=True) if avenant_link_frames else pd.DataFrame()
if not inspection_to_avenant_candidate_links_df.empty:
    dedupe_cols = [c for c in ['linkage_method', 'inspection_key', 'fact_contrat_sk'] if c in inspection_to_avenant_candidate_links_df.columns]
    if dedupe_cols:
        inspection_to_avenant_candidate_links_df = inspection_to_avenant_candidate_links_df.drop_duplicates(dedupe_cols)

compact_export(inspection_to_avenant_candidate_links_df, 'inspection_to_avenant_candidate_links.csv')
display(inspection_to_avenant_candidate_links_df.head(30))

if not inspection_to_avenant_candidate_links_df.empty:
    scenario_b_summary_df = (
        inspection_to_avenant_candidate_links_df.groupby(['linkage_method', 'linkage_quality', 'delay_band'], dropna=False)
        .agg(candidate_links=('fact_contrat_sk', 'count'), inspections=('inspection_key', pd.Series.nunique), contracts=('contrat_sk', pd.Series.nunique), links_with_defects=('has_documented_inspection_defect', 'sum'), links_with_observable_coverage_change=('coverage_change_observable_flag', 'sum'))
        .reset_index()
    )
else:
    scenario_b_summary_df = pd.DataFrame(columns=['linkage_method', 'linkage_quality', 'delay_band', 'candidate_links', 'inspections', 'contracts', 'links_with_defects', 'links_with_observable_coverage_change'])

display(scenario_b_summary_df)

[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\contract_avenant_readiness_summary.csv


,run_ts_utc,metric,value,note
0,2026-07-08 10:21:02+0000,fact_contrat_rows,585514,Contract movement fact rows
1,2026-07-08 10:21:02+0000,valid_contrat_sk,585514,contrat_sk nonzero
2,2026-07-08 10:21:02+0000,missing_contrat_sk_zero_or_null,0,Technical key 0 counted as missing
3,2026-07-08 10:21:02+0000,avenant_rows_est_avenant,138227,Rows flagged as endorsement
4,2026-07-08 10:21:02+0000,update_rows_est_mise_a_jour,424159,Rows flagged as updates
5,2026-07-08 10:21:02+0000,valid_avenant_or_effect_date,585510,"date_derniere_operation_sk, falling back to da..."
6,2026-07-08 10:21:02+0000,product_changed_vs_previous_rows,7971,"Observable product change, not guarantee-level..."
7,2026-07-08 10:21:02+0000,prime_changed_vs_previous_rows,0,"Observable prime change, not guarantee-level p..."
8,2026-07-08 10:21:02+0000,dim_produit_rows_loaded,37,Product labels available if table exists
9,2026-07-08 10:21:02+0000,staging_stg_production_rows_loaded,585514,Raw production rows loaded if table exists


[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\inspection_to_avenant_candidate_links.csv


,linkage_method,linkage_quality,inspection_key,immatriculation_norm,vehicule_sk,fact_contrat_sk,contrat_sk,contrat_mouvement_key,contrat_key,numero_contrat,...,is_avenant,is_update,product_changed_vs_previous,prime_changed_vs_previous,coverage_change_observable_flag,has_documented_inspection_defect,defective_checkpoint_count,critical_defective_checkpoint_count,defective_area,defective_checkpoint_label
0,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,2509TU121|20251208|139,2509TU121,20563,502781,372551,202350000115012|2|1,202350000115012,202350000115012,...,True,True,False,False,False,True,6.0,1.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | SOUS_VEHI...,Controle du niveau du liquide de refroidi | Co...
1,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,3083TU215|20260126|184,3083TU215,28324,489209,362293,202350000101731|0|5,202350000101731,202350000101731,...,False,True,False,False,False,True,1.0,1.0,SOUS_VEHICULE,Controle des plaquettes de freins ar s
2,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,9674TU186|20260330|235,9674TU186,118333,423494,309493,202150000028476|0|9,202150000028476,202150000028476,...,False,True,False,False,False,True,20.0,11.0,ENTRETIEN | INTERIEUR | SOUS_CAPOT | SOUS_VEHI...,Balais essuie glace | Controle des plaquettes ...
3,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,8251TU218|20260107|169,8251TU218,99048,479642,354737,202350000001050|0|3,202350000001050,202350000001050,...,False,True,False,False,False,True,5.0,1.0,INTERIEUR | SOUS_VEHICULE | TOUR_DU_VEHICULE,Balais essuie glace | Controle des plaquettes ...
4,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,3027TU217|20260415|261,3027TU217,27567,517174,385277,202450000006050|0|4,202450000006050,202450000006050,...,False,True,False,False,False,True,5.0,2.0,SOUS_CAPOT | SOUS_VEHICULE | TOUR_DU_VEHICULE,Controle du niveau du liquide de refroidi | Co...
5,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,2847TU174|20260102|162,2847TU174,25170,549029,413591,202550000000531|0|1,202550000000531,202550000000531,...,False,True,False,False,False,True,2.0,1.0,SOUS_VEHICULE | TOUR_DU_VEHICULE,Controle etancheite tous fluides | Retroviseur...
6,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,6492TU160|20260303|211,6492TU160,74947,446402,328279,202250000007051|0|4,202250000007051,202250000007051,...,False,True,False,False,False,True,5.0,5.0,SOUS_VEHICULE,Controle des plaquettes de freins ar s | Contr...
7,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,1941TU86|20251023|95,1941TU86,12776,130410,91558,201750000038469|1|13,201750000038469,201750000038469,...,True,True,False,False,False,True,4.0,1.0,ENTRETIEN | INTERIEUR | SOUS_VEHICULE,Controle fonctionnement des feux de 1 | Contro...
8,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,RS227103|20260408|254,RS227103,126980,425651,311166,202150000030676|2|2,202150000030676,202150000030676,...,True,True,False,False,False,False,NaN,NaN,NaN,NaN
9,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,7460TU200|20260218|201,7460TU200,88279,488222,361545,202350000100823|1|2,202350000100823,202350000100823,...,True,True,False,False,False,True,3.0,2.0,INTERIEUR | SOUS_VEHICULE | TOUR_DU_VEHICULE,Controle avertisseur sonore | Controle des pla...


,linkage_method,linkage_quality,delay_band,candidate_links,inspections,contracts,links_with_defects,links_with_observable_coverage_change
0,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,0-7,4,4,4,4,0
1,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,31-90,33,31,31,28,0
2,IMMATRICULATION_TO_CLAIM_CONTRACT_WEAK,LOW,8-30,14,14,14,12,0


## Readiness decision

Decision logic is conservative. Missing data lowers confidence. Weak links do not produce attention points.

In [24]:
def scenario_a_decision() -> tuple[str, str, str]:
    if inspection_to_claim_candidate_links_df.empty:
        return (
            'BLOCKED',
            'No inspection-to-claim candidate links within 90 days were measured.',
            'Audit raw immatriculation availability and any vehicle-contract bridge before implementation.',
        )
    strong_or_medium = inspection_to_claim_candidate_links_df['linkage_quality'].isin(['HIGH', 'MEDIUM']).any()
    has_defects = inspection_to_claim_candidate_links_df.get('has_documented_inspection_defect', pd.Series(False, index=inspection_to_claim_candidate_links_df.index)).fillna(False).any()
    area_fields_available = any(
        field in claims.columns and claims[field].notna().any()
        for field in ['libelle_garantie', 'cause_sinistre', 'libelle_cause_sinistre', 'motif_cloture_garantie', 'etat_garantie_sinistre']
    )
    if strong_or_medium and has_defects and area_fields_available:
        return (
            'PARTIAL',
            'Candidate links and defect zones exist, but claim damage-area mapping is not yet validated.',
            'Validate a zone/nature mapping before creating mart.fact_post_inspection_attention_signal.',
        )
    return (
        'PARTIAL',
        'Some candidate links exist, but defect or claim-area evidence is incomplete.',
        'Improve linkage confidence and validate damage-area fields before implementation.',
    )


def scenario_b_decision() -> tuple[str, str, str]:
    if inspection_to_avenant_candidate_links_df.empty:
        return (
            'BLOCKED',
            'No inspection-to-avenant candidate links within 90 days were measured.',
            'Find or build a defensible inspection-to-contract linkage path before implementation.',
        )
    has_non_low = inspection_to_avenant_candidate_links_df['linkage_quality'].isin(['HIGH', 'MEDIUM']).any()
    has_observable_change = inspection_to_avenant_candidate_links_df.get('coverage_change_observable_flag', pd.Series(False, index=inspection_to_avenant_candidate_links_df.index)).fillna(False).any()
    if has_non_low and has_observable_change:
        return (
            'PARTIAL',
            'Contract movements and observable product/prime changes exist, but guarantee-level coverage relevance is not validated.',
            'Validate product/coverage change mapping and keep weak links as confidence limitations.',
        )
    return (
        'PARTIAL',
        'Candidate movement links exist mainly through weak or incomplete paths.',
        'Audit direct vehicle-contract linkage and guarantee/product change fields before implementation.',
    )


scenario_a_status, scenario_a_reason, scenario_a_next = scenario_a_decision()
scenario_b_status, scenario_b_reason, scenario_b_next = scenario_b_decision()

decision_df = pd.DataFrame([
    {
        'run_ts_utc': RUN_TS,
        'scenario_code': 'A_INSPECTION_TO_CLAIM_SAME_AREA',
        'scenario_label': 'Inspection to claim on candidate related area',
        'readiness': scenario_a_status,
        'candidate_links_0_90d': len(inspection_to_claim_candidate_links_df),
        'reason': scenario_a_reason,
        'recommended_next_action': scenario_a_next,
    },
    {
        'run_ts_utc': RUN_TS,
        'scenario_code': 'B_INSPECTION_TO_AVENANT_COVERAGE_CHANGE',
        'scenario_label': 'Inspection to endorsement or coverage movement',
        'readiness': scenario_b_status,
        'candidate_links_0_90d': len(inspection_to_avenant_candidate_links_df),
        'reason': scenario_b_reason,
        'recommended_next_action': scenario_b_next,
    },
])

export_report(decision_df, 'post_inspection_readiness_decision.csv')
display(decision_df)

display(Markdown(
    f"""
### Computed readiness summary

- Scenario A readiness: `{scenario_a_status}`
- Scenario B readiness: `{scenario_b_status}`
- Recommended next action: review the generated CSV reports, then validate the accepted linkage method and area/coverage mapping before any mart implementation.
    """.strip()
))

[EXPORT] C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\post_inspection_signals\readiness\post_inspection_readiness_decision.csv


,run_ts_utc,scenario_code,scenario_label,readiness,candidate_links_0_90d,reason,recommended_next_action
0,2026-07-08 10:21:02+0000,A_INSPECTION_TO_CLAIM_SAME_AREA,Inspection to claim on candidate related area,PARTIAL,122,"Candidate links and defect zones exist, but cl...",Validate a zone/nature mapping before creating...
1,2026-07-08 10:21:02+0000,B_INSPECTION_TO_AVENANT_COVERAGE_CHANGE,Inspection to endorsement or coverage movement,PARTIAL,51,Candidate movement links exist mainly through ...,Audit direct vehicle-contract linkage and guar...


### Computed readiness summary

- Scenario A readiness: `PARTIAL`
- Scenario B readiness: `PARTIAL`
- Recommended next action: review the generated CSV reports, then validate the accepted linkage method and area/coverage mapping before any mart implementation.

## Final Readiness Decision

- Scenario A readiness: `GO / PARTIAL / BLOCKED` from `post_inspection_readiness_decision.csv`.
- Scenario B readiness: `GO / PARTIAL / BLOCKED` from `post_inspection_readiness_decision.csv`.
- Recommended next action: keep this module read-only until the linkage method and the damage-area / coverage-change mapping are validated. Do not add these signals to Claim Attention Score V1 before that validation.